# 안전귀가Navi — Day 1 데이터 검증 노트북

**목적:** `data/raw/`에 받아둔 5종 공공데이터의 컬럼·좌표계·결측치·서울 커버리지를 확인하고 Day 2 전처리에 필요한 매핑을 확정.

**검증 대상:**
1. 전국CCTV표준데이터 (`cctcv_seoul.csv`)
2. 서울시 가로등 위치 정보 OA-22205 (`streetlight_seoul.csv`)
3. 행정안전부 안전비상벨 (`emergency_bell.csv`)
4. 안심귀갓길 안전시설물 OA-21696 (`safe_route_facilities_shp.zip` + `safe_route_facility_codes.xlsx`)
5. 안심귀갓길 경로 OA-21695 (`safe_route_path_shp.zip` + `safe_route_path_codes.xlsx`)


In [1]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd

# Windows 콘솔 한글 출력 안정화 (mojibake 방지)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

RAW = Path(r'C:\Users\pjhic\Projects\Seoul_bigdata\data\raw')

print('파일 목록:')
for f in sorted(RAW.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:<40} {size_kb:>10,.1f} KB')


파일 목록:
  cctcv_seoul.csv                            11,400.9 KB
  emergency_bell.csv                          5,705.0 KB
  safe_route_facilities_shp.zip                 471.6 KB
  safe_route_facility_codes.xlsx                 10.5 KB
  safe_route_path_codes.xlsx                     10.3 KB
  safe_route_path_shp.zip                        76.9 KB
  streetlight_seoul.csv                         646.7 KB


In [2]:
def try_read_csv(path: Path) -> tuple[pd.DataFrame, str]:
    """한국 공공데이터에 흔한 인코딩들을 순차 시도."""
    for enc in ['cp949', 'utf-8-sig', 'utf-8', 'euc-kr']:
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False), enc
        except UnicodeDecodeError:
            continue
    raise RuntimeError(f'No encoding worked for {path}')


## 1. 전국CCTV표준데이터

- 메인 감시 인프라 데이터. WGS84 위경도 직접 포함.
- 전국 35만건 → 서울만 필터링 가능한지 확인.

In [3]:
cctv_path = next(RAW.glob('cct*v*seoul*.csv'))
cctv, enc = try_read_csv(cctv_path)
print(f'파일: {cctv_path.name} | 인코딩: {enc}')
print(f'행 수: {len(cctv):,} | 컬럼 수: {len(cctv.columns)}')
print(f'컬럼:\n  {list(cctv.columns)}')


파일: cctcv_seoul.csv | 인코딩: cp949
행 수: 57,931 | 컬럼 수: 18
컬럼:
  ['개방자치단체코드', '관리번호', '관리기관명', '소재지도로명주소', '소재지지번주소', '설치목적구분', '카메라대수', '카메라화소수', '촬영방면정보', '보관일수', '설치연월', '관리기관전화번호', 'WGS84위도', 'WGS84경도', '데이터기준일자', '데이터갱신구분', '데이터갱신시점', '최종수정시점']


In [4]:
cctv.head(3)

,개방자치단체코드,관리번호,관리기관명,소재지도로명주소,소재지지번주소,설치목적구분,카메라대수,카메라화소수,촬영방면정보,보관일수,설치연월,관리기관전화번호,WGS84위도,WGS84경도,데이터기준일자,데이터갱신구분,데이터갱신시점,최종수정시점
0,3000000,2.025300e+17,서울특별시 종로구청,서울특별시 종로구 자하문로17길 36,NaN,생활방범,4,200.0,360도 전방면,30.0,202301.0,02-2148-3033,37.58140,126.9686,2025-10-31,I,2026-03-24 3:27,2025-10-23 14:47
1,3000000,2.025300e+17,서울특별시 종로구청,서울특별시 종로구 종로18길 21,NaN,생활방범,3,200.0,360도 전방면,30.0,201412.0,02-2148-3033,37.56930,126.9914,2025-10-31,I,2026-03-24 3:27,2025-10-23 14:47
2,3000000,2.025300e+17,서울특별시 종로구청,서울특별시 종로구 송월1길 44-22,NaN,생활방범,2,200.0,360도 전방면,30.0,201312.0,02-2148-3033,37.57181,126.9656,2025-10-31,I,2026-03-24 3:27,2025-10-23 14:47


In [5]:
# 결측치 + dtype 확인
info = pd.DataFrame({
    'dtype': cctv.dtypes.astype(str),
    'null': cctv.isna().sum(),
    'null_pct': (cctv.isna().mean() * 100).round(1),
})
info

,dtype,null,null_pct
개방자치단체코드,int64,0,0.0
관리번호,float64,0,0.0
관리기관명,object,0,0.0
소재지도로명주소,object,5932,10.2
소재지지번주소,object,9983,17.2
설치목적구분,object,0,0.0
카메라대수,int64,0,0.0
카메라화소수,float64,16461,28.4
촬영방면정보,object,32959,56.9
보관일수,float64,7522,13.0


In [6]:
# 서울 데이터 비율 확인 — 도로명주소 컬럼으로 필터
addr_col = '소재지도로명주소'
if addr_col not in cctv.columns:
    addr_col = next((c for c in cctv.columns if '도로명' in c), None)

cctv_is_seoul = cctv[addr_col].astype(str).str.startswith('서울')
print(f'서울 행: {cctv_is_seoul.sum():,} / 전체 {len(cctv):,} ({cctv_is_seoul.mean()*100:.1f}%)')

# 자치구 분포
cctv_seoul = cctv[cctv_is_seoul].copy()
cctv_seoul['자치구'] = cctv_seoul[addr_col].str.extract(r'서울특별시\s+([가-힣]+구)')
print(f'\n자치구 커버리지 ({cctv_seoul["자치구"].nunique()}/25개):')
print(cctv_seoul['자치구'].value_counts())

# 누락 자치구 식별
SEOUL_GU = {'종로구','중구','용산구','성동구','광진구','동대문구','중랑구','성북구','강북구','도봉구',
            '노원구','은평구','서대문구','마포구','양천구','강서구','구로구','금천구','영등포구','동작구',
            '관악구','서초구','강남구','송파구','강동구'}
covered = set(cctv_seoul['자치구'].dropna().unique())
missing = SEOUL_GU - covered
print(f'\n누락 자치구: {missing if missing else "없음"}')


서울 행: 48,765 / 전체 57,931 (84.2%)

자치구 커버리지 (23/25개):
자치구
영등포구    5300
양천구     4795
강북구     4360
서초구     3239
노원구     3111
도봉구     2807
성북구     2346
구로구     2132
용산구     1930
강서구     1805
강남구     1793
중랑구     1791
관악구     1530
마포구     1497
동대문구    1324
광진구     1304
서대문구    1292
송파구     1190
중구      1136
종로구     1040
금천구      935
동작구      857
성동구        1
Name: count, dtype: int64

누락 자치구: {'강동구', '은평구'}


In [7]:
# 좌표 유효성 — WGS84 위경도 범위 체크 (서울 대략 위도 37.4~37.7, 경도 126.7~127.2)
lat_col = next(c for c in cctv.columns if '위도' in c)
lon_col = next(c for c in cctv.columns if '경도' in c)
print(f'좌표 컬럼: {lat_col}, {lon_col}')
print(cctv_seoul[[lat_col, lon_col]].describe())

# 서울 범위 밖 좌표 검출
out_of_seoul = ~(
    cctv_seoul[lat_col].between(37.4, 37.75)
    & cctv_seoul[lon_col].between(126.7, 127.2)
)
print(f'\n서울 좌표 범위 밖 의심: {out_of_seoul.sum():,}건')

좌표 컬럼: WGS84위도, WGS84경도
            WGS84위도       WGS84경도
count  48765.000000  48765.000000
mean      37.555469    126.978761
std        0.058384      0.415868
min       37.030010    126.798597
25%       37.507980    126.904300
50%       37.544330    127.000300
75%       37.601700    127.041600
max       37.694300    216.994700

서울 좌표 범위 밖 의심: 14건


## 2. 서울시 가로등 위치 정보 (OA-22205)

- 야간 조명 점수의 핵심 데이터.
- 컬럼이 매우 단순(관리번호 + 위도 + 경도)할 것으로 예상.

In [8]:
sl, enc = try_read_csv(RAW / 'streetlight_seoul.csv')
print(f'인코딩: {enc} | 행: {len(sl):,} | 컬럼: {list(sl.columns)}')
sl.head()

인코딩: cp949 | 행: 19,316 | 컬럼: ['관리번호', '위도', '경도']


,관리번호,위도,경도
0,가락지하차도-01,37.495254,127.107417
1,가락지하차도-02,37.495354,127.107755
2,가락지하차도-03,37.495443,127.108073
3,가락지하차도-04,37.495551,127.108410
4,가락지하차도-05,37.495654,127.108736


In [9]:
# 좌표 유효성 + 서울 범위
lat_col = next(c for c in sl.columns if '위도' in c or 'lat' in c.lower())
lon_col = next(c for c in sl.columns if '경도' in c or 'lon' in c.lower())
print(sl[[lat_col, lon_col]].describe())

out = ~(sl[lat_col].between(37.4, 37.75) & sl[lon_col].between(126.7, 127.2))
print(f'\n서울 범위 밖 가로등: {out.sum():,}건 ({out.mean()*100:.1f}%)')
print(f'결측 좌표: lat {sl[lat_col].isna().sum()}, lon {sl[lon_col].isna().sum()}')

                 위도            경도
count  19316.000000  1.931600e+04
mean      37.496440  6.163343e+03
std        1.350365  1.676843e+05
min        0.000000  1.267983e+02
25%       37.520020  1.269202e+02
50%       37.537847  1.270195e+02
75%       37.566135  1.270732e+02
max       37.689633  4.664046e+06

서울 범위 밖 가로등: 25건 (0.1%)
결측 좌표: lat 0, lon 0


In [10]:
# 관리번호 prefix로 자치구/관리주체 추정 가능한지 확인
id_col = next(c for c in sl.columns if '관리' in c or 'ID' in c)
print(f'관리번호 prefix 분포 (앞 10자):')
print(sl[id_col].astype(str).str[:10].value_counts().head(20))

관리번호 prefix 분포 (앞 10자):
관리번호
가양대교램프_01-    178
서강대교북단_01-     70
마포대교북단_상_0     64
동작대교북단_01-     62
성수대교남단_03-     61
동작대교북단_02-     56
동작대교남단_01-     54
성수대교남단_04-     54
잠실대교북단_01-     53
동작대교남단_02-     50
한남대교남단_하-0     49
성수대교남단_01-     48
올림픽대교북단_02     47
성수대교북단_02-     47
구리암사대교남단상류     46
천호대교남단_01-     45
한남대교북단_하-0     45
성수대교북단_03-     43
한남대교북단_상-0     42
한남대교남단_상-0     41
Name: count, dtype: int64


## 3. 행정안전부 안전비상벨

- 긴급 대응 점수의 핵심 데이터.
- 전국 데이터 → 서울만 필터링.

In [11]:
eb, enc = try_read_csv(RAW / 'emergency_bell.csv')
print(f'인코딩: {enc} | 행: {len(eb):,} | 컬럼 수: {len(eb.columns)}')
print(f'컬럼:\n  {list(eb.columns)}')

인코딩: cp949 | 행: 23,378 | 컬럼 수: 24
컬럼:
  ['개방자치단체코드', '관리번호', '안전비상벨관리번호', '설치목적', '설치장소유형', '설치위치', '소재지도로명주소', '소재지지번주소', 'WGS84위도', 'WGS84경도', '연계방식', '경찰연계유무', '경비업체연계유무', '관리사무소연계유무', '부가기능', '안전비상벨설치연도', '최종점검일자', '최종점검결과구분', '관리기관명', '관리기관전화번호', '데이터기준일자', '데이터갱신구분', '데이터갱신시점', '최종수정시점']


In [12]:
eb.head(3)

,개방자치단체코드,관리번호,안전비상벨관리번호,설치목적,설치장소유형,설치위치,소재지도로명주소,소재지지번주소,WGS84위도,WGS84경도,연계방식,경찰연계유무,경비업체연계유무,관리사무소연계유무,부가기능,안전비상벨설치연도,최종점검일자,최종점검결과구분,관리기관명,관리기관전화번호,데이터기준일자,데이터갱신구분,데이터갱신시점,최종수정시점
0,3000000,2.024300e+17,마로니에공원-001호,방범용,공원,마로니에공원,서울특별시 종로구 대학로 104,동숭동 1-124,37.580336,127.002513,양방향,N,N,N,CCTV,2018,2020-06-30,Y,종로구청,02-2148-2844,2020-07-01,I,2025-12-15 16:21,2024-07-31 9:52
1,3000000,2.024300e+17,명륜상상어린이공원-001호,방범용,공원,명륜상상어린이공원,서울특별시 종로구 혜화로9길 24-9,명륜1가 23-9,37.588055,126.998283,양방향,N,N,N,CCTV,2018,2020-06-30,Y,종로구청,02-2148-2844,2020-07-01,I,2025-12-15 16:21,2024-07-31 9:52
2,3000000,2.024300e+17,삼청공원-003호,약자보호,화장실,삼청공원(약수터 위),서울특별시 종로구 북촌로 134-3,삼청동 산2-1,37.587979,126.984085,양방향,Y,N,N,NaN,2018,2020-06-30,Y,종로구청,02-2148-2843,2020-07-01,I,2025-12-15 16:21,2024-07-31 9:52


In [13]:
# 서울 필터
addr_col = next(c for c in eb.columns if '도로명' in c)
eb_is_seoul = eb[addr_col].astype(str).str.startswith('서울')
print(f'서울 안전비상벨: {eb_is_seoul.sum():,} / {len(eb):,} ({eb_is_seoul.mean()*100:.1f}%)')

eb_seoul = eb[eb_is_seoul].copy()
eb_seoul['자치구'] = eb_seoul[addr_col].str.extract(r'서울특별시\s+([가-힣]+구)')
print(f'\n자치구 커버리지 ({eb_seoul["자치구"].nunique()}/25개):')
print(eb_seoul['자치구'].value_counts())

covered = set(eb_seoul['자치구'].dropna().unique())
missing = SEOUL_GU - covered
print(f'\n누락 자치구: {missing if missing else "없음"}')


서울 안전비상벨: 20,763 / 23,378 (88.8%)



자치구 커버리지 (24/25개):
자치구
구로구     1442
용산구     1421
성북구     1420
중랑구     1351
영등포구    1307
마포구     1268
강남구     1262
광진구     1119
도봉구     1053
관악구     1000
양천구      976
강동구      935
강서구      930
동대문구     833
중구       728
서대문구     723
금천구      709
동작구      627
은평구      611
노원구       63
종로구       57
서초구       52
강북구       22
송파구        1
Name: count, dtype: int64

누락 자치구: {'성동구'}


In [14]:
# 비상벨 카테고리 분포 — 긴급대응 점수 가중치 설계에 활용
for col in ['설치장소유형', '부가기능', '운영여부']:
    if col in eb_seoul.columns:
        print(f'\n[{col}] 분포:')
        print(eb_seoul[col].value_counts().head(10))


[설치장소유형] 분포:
설치장소유형
가로변    16041
기타      2312
공원      1984
화장실      302
주차장       83
건물        41
Name: count, dtype: int64

[부가기능] 분포:
부가기능
CCTV            3877
경보음+CCTV         709
경보등+경보음          146
X                 22
경보음               20
CCTV+경보음+경고등       1
Name: count, dtype: int64


## 4. 안심귀갓길 안전시설물 (OA-21696, SHP)

- POINT geometry, EPSG:4326(WGS84) 사용 예상.
- 시설물 유형: CCTV / 안심벨 / 안내표지판.

In [15]:
fac = gpd.read_file(RAW / 'safe_route_facilities_shp.zip')
print(f'행: {len(fac):,} | CRS: {fac.crs} | geometry: {fac.geometry.geom_type.unique()}')
print(f'컬럼:\n  {list(fac.columns)}')

행: 11,883 | CRS: EPSG:4326 | geometry: ['Point']
컬럼:
  ['포인트 WKT', '시설물 ID', '시군구 코', '시군구명', '읍면동 코', '읍면동명', '시설코드', '안심귀갓길', '안심귀갓_1', '설치대수', '비고', '관리기관', '전화번호', '조성년월', '세부위치', '데이터 기', '이미지명', 'geometry']


In [16]:
fac.drop(columns='geometry').head(3)

,포인트 WKT,시설물 ID,시군구 코,시군구명,읍면동 코,읍면동명,시설코드,안심귀갓길,안심귀갓_1,설치대수,비고,관리기관,전화번호,조성년월,세부위치,데이터 기,이미지명
0,POINT (126.968590563668 37.5793826677127),1111011000_04_P01,1111000000,서울특별시 종로구,1111011000,누하동,301,1111011000_04,종로안심04,1,None,종로CCTV통합안전센터,02-2148-4301,2015,None,20221109,1111011000_04_P01_301.jpeg
1,POINT (126.968590563668 37.5793826677127),1111011000_04_P02,1111000000,서울특별시 종로구,1111011000,누하동,307,1111011000_04,종로안심04,1,None,서울종로경찰서,None,2015,None,20221109,1111011000_04_P02_307.jpeg
2,POINT (126.968590563668 37.5793826677127),1111011000_04_P03,1111000000,서울특별시 종로구,1111011000,누하동,305,1111011000_04,종로안심04,1,None,None,None,2015,None,20221109,1111011000_04_P03_305.jpeg


In [17]:
# 시설물 유형은 '시설코드' 숫자값에 들어있음 (코드 사전: 301=안심벨, 302=CCTV, 303=안내표지판...)
FACILITY_CODE_MAP = {
    301: '안심벨',
    302: 'CCTV',
    303: '안내표지판',
    304: '노면표기',
    305: '안심귀갓길 서비스 안내판',
    306: '112 위치신고 안내판',
    307: '보안등',
    308: '기타 시설물',
}
fac['시설유형'] = fac['시설코드'].astype(int).map(FACILITY_CODE_MAP)
print('[시설유형] 분포:')
print(fac['시설유형'].value_counts(dropna=False))


[시설유형] 분포:
시설유형
안심귀갓길 서비스 안내판    5618
보안등              1758
노면표기             1498
CCTV             1487
안심벨              1001
안내표지판             376
기타 시설물             75
112 위치신고 안내판       70
Name: count, dtype: int64


In [18]:
# 자치구 커버리지
sigungu_col = next((c for c in fac.columns if '시군구' in c and '명' in c), None)
if sigungu_col:
    print(f'자치구 커버리지 ({fac[sigungu_col].nunique()}개):')
    print(fac[sigungu_col].value_counts())

자치구 커버리지 (25개):
시군구명
서울특별시 강북구     953
서울특별시 강남구     848
서울특별시 성북구     727
서울특별시 관악구     686
서울특별시 서초구     666
서울특별시 용산구     607
서울특별시 동작구     600
서울특별시 송파구     571
서울특별시 영등포구    546
서울특별시 노원구     528
서울특별시 강동구     493
서울특별시 종로구     462
서울특별시 중랑구     459
서울특별시 서대문구    450
서울특별시 동대문구    439
서울특별시 중구      392
서울특별시 강서구     374
서울특별시 도봉구     324
서울특별시 구로구     309
서울특별시 은평구     291
서울특별시 마포구     274
서울특별시 광진구     270
서울특별시 금천구     210
서울특별시 양천구     204
서울특별시 성동구     200
Name: count, dtype: int64


In [19]:
# 코드 사전 — 헤더 위치 자동 탐색
fac_codes_raw = pd.read_excel(RAW / 'safe_route_facility_codes.xlsx', header=None)
print(f'shape: {fac_codes_raw.shape}')
fac_codes_raw.head(20)

shape: (20, 5)


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,안심귀갓길 안전시설물 통합데이터,NaN,NaN
2,NaN,NaN,항목명,설명,예시
3,NaN,NaN,포인트 WKT,"POINT(경도 위도)\n\n*경도 : X좌표, 동, 세로선\n*위도 : Y좌표, ...",POINT(127.06830674135355252 37.54306698366694661)
4,NaN,NaN,포인트 ID,안심귀갓길 아이디_P01 ~\n\n*각 안심귀갓길 객체마다 01부터 다시 시작,1121510700_01_P01
5,NaN,NaN,시군구 코드,십진수 10자리 (행정표준코드 참조),1121500000
6,NaN,NaN,시군구명,NaN,서울특별시 광진구
7,NaN,NaN,읍면동 코드,십진수 10자리 (행정표준코드 참조),1121510700
8,NaN,NaN,읍면동명,NaN,화양동
9,NaN,NaN,시설코드,숫자만 표기\n\n301 안심벨\n302 CCTV\n303 안내표지판 (전주 표기 ...,305


## 5. 안심귀갓길 경로 (OA-21695, SHP LINESTRING)

- LINESTRING geometry. 안심귀갓길 ID로 시설물(#4)과 조인 가능.
- 입력 주소에서 가장 가까운 경로까지의 거리 계산에 사용.

In [20]:
paths = gpd.read_file(RAW / 'safe_route_path_shp.zip')
print(f'행: {len(paths):,} | CRS: {paths.crs} | geometry: {paths.geometry.geom_type.unique()}')
print(f'컬럼:\n  {list(paths.columns)}')

행: 362 | CRS: EPSG:4326 | geometry: ['LineString' 'MultiLineString']
컬럼:
  ['길이', '시군구 코', '시군구명', '읍면동 코', '읍면동명', '안심벨', 'CCTV', '안심귀갓길', '안심귀갓_1', '보안등', '안심귀갓_2', '112 위치신', '기타 시설', '안심귀갓_3', '안심귀갓_4', '조성년월', '세부위치', '비고', '데이터 기', 'geometry']


In [21]:
paths.drop(columns='geometry').head(5)

,길이,시군구 코,시군구명,읍면동 코,읍면동명,안심벨,CCTV,안심귀갓길,안심귀갓_1,보안등,안심귀갓_2,112 위치신,기타 시설,안심귀갓_3,안심귀갓_4,조성년월,세부위치,비고,데이터 기
0,305,1111000000,서울특별시 종로구,1111011000,누하동,4,13,None,5,14.0,None,4,None,1111011000_04,종로안심04,2015,필운대로5길,None,20221109
1,211.25,1111000000,서울특별시 종로구,1111011200,체부동,1,8,None,3,7.0,None,3,None,1111011200_03,종로안심03,2015,자하문로5길,None,20221109
2,481.92,1111000000,서울특별시 종로구,1111014900,원서동,3,5,None,5,26.0,None,4,None,1111014900_01,종로안심01,2017,창덕궁1가길,None,20221110
3,133.55,1111000000,서울특별시 종로구,1111016800,동숭동,2,2,None,2,7.0,None,3,None,1111016800_04,혜화안심04,2017,동숭4나길,None,20221117
4,247.78,1111000000,서울특별시 종로구,1111016900,혜화동,2,2,None,6,15.0,None,6,None,1111016900_01,혜화안심01,2015,혜화로 6길,None,20221117


In [22]:
# 경로별 길이 (m) — 좌표계가 4326이면 to_crs로 미터 변환 후 length
paths_m = paths.to_crs(epsg=5179)  # UTM-K 한국 표준
lengths = paths_m.geometry.length
print(f'경로 길이 통계 (m):')
print(lengths.describe())
print(f'\n총 안심귀갓길 길이: {lengths.sum()/1000:.1f} km')

경로 길이 통계 (m):
count    362.000000
mean     371.606266
std      130.091025
min      132.312460
25%      272.814704
50%      347.693138
75%      455.920894
max      832.262236
dtype: float64

총 안심귀갓길 길이: 134.5 km


In [23]:
# 자치구 분포
sigungu_col = next((c for c in paths.columns if '시군구' in c and '명' in c), None)
if sigungu_col:
    print(f'안심귀갓길 자치구 분포 ({paths[sigungu_col].nunique()}개):')
    print(paths[sigungu_col].value_counts())

안심귀갓길 자치구 분포 (25개):
시군구명
서울특별시 성북구     24
서울특별시 강남구     24
서울특별시 강북구     22
서울특별시 관악구     19
서울특별시 서초구     19
서울특별시 노원구     18
서울특별시 동작구     18
서울특별시 용산구     17
서울특별시 중랑구     16
서울특별시 강동구     16
서울특별시 서대문구    16
서울특별시 송파구     15
서울특별시 강서구     15
서울특별시 종로구     15
서울특별시 영등포구    15
서울특별시 중구      13
서울특별시 광진구     12
서울특별시 도봉구     12
서울특별시 동대문구    11
서울특별시 마포구     10
서울특별시 은평구     10
서울특별시 구로구      9
서울특별시 성동구      6
서울특별시 금천구      6
서울특별시 양천구      4
Name: count, dtype: int64


In [24]:
path_codes_raw = pd.read_excel(RAW / 'safe_route_path_codes.xlsx', header=None)
print(f'shape: {path_codes_raw.shape}')
path_codes_raw.head(20)

shape: (22, 5)


,0,1,2,3,4
0,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,안심귀갓길 경로 기반 링크 데이터,NaN,NaN
2,NaN,NaN,항목명,설명,예시
3,NaN,NaN,링크 길이,"단위 : 미터(M)\n\n숫자만 입력, 소수점 2자리, 단위까지 입력",26.68
4,NaN,NaN,시군구 코드,십진수 10자리 (행정표준코드 참조),1121500000
5,NaN,NaN,시군구명,NaN,서울특별시 광진구
6,NaN,NaN,읍면동 코드,십진수 10자리 (행정표준코드 참조),1121510700
7,NaN,NaN,읍면동명,NaN,화양동
8,NaN,NaN,안심벨,안심벨의 개수,1
9,NaN,NaN,CCTV,CCTV의 개수,5


## 6. 검증 요약

각 데이터별로 다음을 확정:
- 사용할 좌표 컬럼명
- 자치구 25개 전체 커버 여부
- 결측치 비율
- Day 2 전처리에서 통일할 좌표계 (모두 EPSG:4326이면 변환 불필요)

In [25]:
summary = pd.DataFrame([
    {'데이터': 'CCTV',         '전체 행': len(cctv),  '서울 행': int(cctv_is_seoul.sum()), '자치구 커버': f'{cctv_seoul["자치구"].nunique()}/25', '좌표계': 'WGS84 (위/경도)'},
    {'데이터': '가로등',         '전체 행': len(sl),    '서울 행': len(sl),                   '자치구 커버': '관리주체별(다리 중심)', '좌표계': '위도/경도'},
    {'데이터': '안전비상벨',     '전체 행': len(eb),    '서울 행': int(eb_is_seoul.sum()),    '자치구 커버': f'{eb_seoul["자치구"].nunique()}/25',  '좌표계': 'WGS84 (위/경도)'},
    {'데이터': '안심귀갓길 시설물','전체 행': len(fac),  '서울 행': len(fac),                  '자치구 커버': f'{fac["시군구명"].nunique()}/25', '좌표계': str(fac.crs)},
    {'데이터': '안심귀갓길 경로', '전체 행': len(paths),'서울 행': len(paths),                '자치구 커버': f'{paths["시군구명"].nunique()}/25','좌표계': str(paths.crs)},
])
summary


,데이터,전체 행,서울 행,자치구 커버,좌표계
0,CCTV,57931,48765,23/25,WGS84 (위/경도)
1,가로등,19316,19316,관리주체별(다리 중심),위도/경도
2,안전비상벨,23378,20763,24/25,WGS84 (위/경도)
3,안심귀갓길 시설물,11883,11883,25/25,EPSG:4326
4,안심귀갓길 경로,362,362,25/25,EPSG:4326
